# 05 — Contrôle pré-prod (garde-fou AVANT le build HTML)

Garde-fou **autonome** : vérifie le **format des fichiers à importer** et le **contrat
du fichier produit** `data/donnees.csv`, AVANT de lancer le build des ~500 pages.

> ⚠ **Aucune cellule n'importe `src/`** : on n'utilise que `pandas` / `openpyxl` / `yaml`.
> Le but est de détecter une **dérive de format** des fichiers sources INDÉPENDAMMENT du
> code d'ingestion (`chargeur_long`) — un fichier non conforme est repéré même si le loader
> change ou masque l'erreur.

S'exécute à l'identique en **fictif** (générateur Option B) et en **réel** (chargeur YAML).

1. Chargement (`donnees.csv` brut).
2. Couverture.
3. Contrôle 1 — format des fichiers sources bruts (piloté par `descriptif_sources.yaml`).
4. Contrôle 2 — contrat de `donnees.csv`.
5. Synthèse (tests critiques → échec bloque le build).
6. Exploration libre.

## 1. Chargement

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

DATA_DIR = Path('../data')
YAML_PATH = '../docs/descriptif_sources.yaml'
# dtype string sur stade (majoritairement vide hors survie) -> pas de DtypeWarning
long = pd.read_csv(DATA_DIR / 'donnees.csv', dtype={'stade': 'string'})
print(f"donnees.csv : {len(long):,} lignes - {long['variable'].nunique()} variables - "
      f"sources {sorted(long['source'].unique())}")

# Harness check/tests (ok dans {True, False, None=non verifie})
tests = []
def check(nom, ok, detail='', critique=True):
    tests.append((nom, ok, detail, bool(critique)))

def brancher(resultats, source_presente=True):
    for ok, lib in resultats:
        check(lib, ok, '', critique=(ok is not None and source_presente))

def _tag(ok, crit=True):
    return {True: '✓', False: ('✗' if crit else '⚠'), None: '○'}[ok]

donnees.csv : 174,578 lignes - 12 variables - sources ['BN', 'DIM APHP', 'EDS APHP']


## 2. Couverture

In [2]:
long.pivot_table(index=['source', 'niveau'], columns='variable',
                 values='valeur', aggfunc='count', fill_value=0)

variable            delai_chirurgie_median  delai_global_median  \
source   niveau                                                   
BN       aphp                            0                    0   
         type_etab                       0                    0   
DIM APHP aphp                          288                  288   
         ghu                          1728                 1728   
         hopital                      9216                 9216   
EDS APHP aphp                            0                    0   
         ghu                             0                    0   
         hopital                         0                    0   

variable            delai_radio_median  delai_traitement_medical_median  \
source   niveau                                                           
BN       aphp                        0                                0   
         type_etab                   0                                0   
DIM APHP aphp                      288                              288   
         ghu                      1728                             1728   
         hopital                  9216                             9216   
EDS APHP aphp                        0                                0   
         ghu                         0                                0   
         hopital                     0                                0   

variable            nb_patients  nb_patients_stade  nb_sejours_chimiotherapie  \
source   niveau                                                                 
BN       aphp               576                  0                        288   
         type_etab         2880                  0                       1440   
DIM APHP aphp               576                  0                        288   
         ghu               3456                  0                       1728   
         hopital          18432                  0                       9216   
EDS APHP aphp                 0               1408                          0   
         ghu                  0               8448                          0   
         hopital              0               8192                          0   

variable            nb_sejours_chirurgie  nb_sejours_palliatifs  \
source   niveau                                                   
BN       aphp                        288                    288   
         type_etab                  1440                   1440   
DIM APHP aphp                        288                    288   
         ghu                        1728                   1728   
         hopital                    9216                   9216   
EDS APHP aphp                          0                      0   
         ghu                           0                      0   
         hopital                       0                      0   

variable            nb_sejours_radiotherapie  survie_1an  survie_5ans  
source   niveau                                                        
BN       aphp                            288           0            0  
         type_etab                      1440           0            0  
DIM APHP aphp                            288           0            0  
         ghu                            1728           0            0  
         hopital                        9216           0            0  
EDS APHP aphp                              0        1352         1352  
         ghu                               0        7656         7656  
         hopital                           0        7913         7913

In [3]:
print('Entités par niveau :')
for n in sorted(long['niveau'].unique()):
    print(f"  {n:<10} : {long[long.niveau == n].entite.nunique():>3} entités")
print('\nAnnées par source :')
for s, g in long.groupby('source'):
    print(f"  {s:<9} : {sorted(int(a) for a in g.annee.unique())}")
nbst = len(long[long.variable == 'nb_patients_stade'])
masq = nbst - len(long[long.variable == 'survie_1an'])
print(f"\nSurvie masquée (effectifs < 5) : {masq} / {nbst} = {masq/max(1,nbst):.1%}")

Entités par niveau :
  aphp       :   1 entités
  ghu        :   6 entités
  hopital    :  32 entités
  type_etab  :   5 entités

Années par source :
  BN        : [2022, 2023, 2024, 2025]
  DIM APHP  : [2022, 2023, 2024, 2025]
  EDS APHP  : [2022, 2023, 2024, 2025]



Survie masquée (effectifs < 5) : 1127 / 18048 = 6.2%


## 3. Contrôle 1 — Format des fichiers sources bruts

Piloté par `descriptif_sources.yaml`. Pour chaque source (OECI **et** régional) et chaque
feuille déclarée : présence, libellés de dimensions aux positions attendues, modalités de
`Niveau` reconnues, forme GHU, 4 blocs de délais (coquilles tolérées par mots-clés).
**Fichier/feuille absent → informatif** (○, jamais un échec) ; le contrôle s'active dès que
les vrais fichiers arrivent.

In [4]:
import glob, os
import openpyxl

_KW_DIM = {'niveau': 'niveau', 'ghu': 'ghu', 'hopital': 'pital', 'appareil': 'appareil',
           'organe': 'organe', 'hopital_aphp': 'ap-hp', 'finess': 'finess',
           'statut': 'statut', 'raison': 'raison', 'annee': 'ann'}

def _low(x):
    return str(x if x is not None else '').strip().lower()

def _agg_niveau(niv, cn):
    if isinstance(cn, dict) and cn.get('mode') == 'mots_cles':
        u = str(niv or '').upper()
        if 'HOP' in u or 'HÔP' in u: return 'hopital'
        if 'GH' in u: return 'ghu'
        if 'AP-HP' in u or 'APHP' in u: return 'aphp'
        return None
    e = cn.get(niv) if isinstance(cn, dict) else None
    return e.get('agg') if isinstance(e, dict) else None

def _niveau_reconnu(niv, cn):
    if isinstance(cn, dict) and cn.get('mode') == 'mots_cles':
        return _agg_niveau(niv, cn) is not None
    return niv in cn

def _fichier_reel(dossier, motif):
    cands = [p for p in glob.glob(os.path.join(str(dossier), motif))
             if '_fictif' not in os.path.basename(p)]
    return sorted(cands)[0] if cands else None

def _controler_feuille(ws, conf, prefixe, res):
    nom, dims, ne = conf['nom'], conf['dimensions'], conf['lignes_entete']
    r0, cn = conf['premiere_ligne_donnees'], conf['niveau']
    bad = [f"{d}@col{p}={ws.cell(ne, p).value!r}" for d, p in dims.items()
           if _KW_DIM.get(d, d) not in _low(ws.cell(ne, p).value)]
    res.append((not bad, f"{prefixe} « {nom} » : libellés de dimensions"
                + (f" — ANOMALIE {bad}" if bad else "")))
    obs = set()
    for r in range(r0, min(ws.max_row, r0 + 3000) + 1):
        v = ws.cell(r, dims['niveau']).value
        if v not in (None, ''): obs.add(str(v).strip())
    inconnus = sorted(v for v in obs if not _niveau_reconnu(v, cn))
    res.append((not inconnus, f"{prefixe} « {nom} » : modalités Niveau reconnues"
                + (f" — INCONNUES {inconnus}" if inconnus else "")))
    if 'ghu' in dims and conf.get('ghu_forme'):
        forme, vals = conf['ghu_forme'], set()
        for r in range(r0, min(ws.max_row, r0 + 3000) + 1):
            if _agg_niveau(ws.cell(r, dims['niveau']).value, cn) == 'ghu':
                g = ws.cell(r, dims['ghu']).value
                if g not in (None, ''): vals.add(str(g).strip())
        if vals:
            if forme == 'court':
                ok = all(_low(v).startswith('ap-hp.') or _low(v).startswith('ap-hp,') for v in vals)
            else:
                ok = all('.' in v for v in vals)
            res.append((ok, f"{prefixe} « {nom} » : forme GHU = {forme}"
                        + ('' if ok else f" — INATTENDU {sorted(vals)[:3]}")))
    if conf['mesures']['disposition'] == 'blocs':
        txt = ' '.join(_low(ws.cell(1, col).value) for col in range(1, ws.max_column + 1))
        manque = []
        if 'total' not in txt: manque.append('TOTAL')
        if 'chir' not in txt: manque.append('CHIRURGIE')
        if 'med' not in txt and 'méd' not in txt: manque.append('MEDECINE')
        if not any(k in txt for k in ('radio', 'rafio', 'therap')): manque.append('RADIOTHERAPIE')
        res.append((not manque, f"{prefixe} « {nom} » : 4 blocs de délais (coquilles tolérées)"
                    + (f" — MANQUE {manque}" if manque else "")))

def controler_sources_brutes(dossier=DATA_DIR, yaml_path=YAML_PATH):
    desc = yaml.safe_load(open(yaml_path, encoding='utf-8'))
    res = []
    o = desc['sources']['oeci']
    path = _fichier_reel(dossier, o['fichier'])
    if path is None:
        res.append((None, f"OECI ({o['fichier']}) — aucun fichier réel dans data/ (non vérifié)"))
    else:
        res.append((True, f"OECI : fichier réel = {os.path.basename(path)}"))
        wb = openpyxl.load_workbook(path, data_only=True)
        for conf in o['feuilles'].values():
            if 'mesures' not in conf:
                pres = conf['nom'] in wb.sheetnames
                res.append((None, f"OECI feuille « {conf['nom']} » (non consommée) — "
                            + ('présente' if pres else 'absente')))
            elif conf['nom'] not in wb.sheetnames:
                oblig = bool(conf.get('obligatoire'))
                res.append((False if oblig else None, f"OECI feuille « {conf['nom']} » absente"
                            + (' (OBLIGATOIRE)' if oblig else ' (optionnelle)')))
            else:
                _controler_feuille(wb[conf['nom']], conf, 'OECI', res)
        wb.close()
    rg = desc['sources']['regional']
    for conf in rg['feuilles'].values():
        motif = rg['fichiers'][conf['fichier']]
        path = _fichier_reel(dossier, motif)
        if path is None:
            res.append((None, f"Régional ({motif}) — absent dans data/ (non vérifié)"))
            continue
        res.append((True, f"Régional : fichier réel = {os.path.basename(path)}"))
        wb = openpyxl.load_workbook(path, data_only=True)
        if conf['nom'] not in wb.sheetnames:
            res.append((False, f"Régional {conf['fichier']} : feuille « {conf['nom']} » absente"))
        else:
            _controler_feuille(wb[conf['nom']], conf, 'Régional', res)
        wb.close()
    return res

_src = controler_sources_brutes()
print(f"Contrôle 1 — {len(_src)} vérification(s) :")
for ok, lib in _src:
    print(f"  {_tag(ok)} {lib}")
brancher(_src)

Contrôle 1 — 19 vérification(s) :
  ✓ OECI : fichier réel = indicateurs_oeci_2025_M12.xlsx
  ✓ OECI « Indicateurs patient » : libellés de dimensions
  ✓ OECI « Indicateurs patient » : modalités Niveau reconnues
  ✓ OECI « Indicateurs patient » : forme GHU = long
  ✓ OECI « Indicateurs séjour » : libellés de dimensions
  ✓ OECI « Indicateurs séjour » : modalités Niveau reconnues
  ✓ OECI « Indicateurs séjour » : forme GHU = long
  ✓ OECI « Survie globale » : libellés de dimensions
  ✓ OECI « Survie globale » : modalités Niveau reconnues
  ✓ OECI « Survie globale » : forme GHU = court
  ✓ OECI « Délais PEC » : libellés de dimensions
  ✓ OECI « Délais PEC » : modalités Niveau reconnues
  ✓ OECI « Délais PEC » : forme GHU = long
  ✓ OECI « Délais PEC » : 4 blocs de délais (coquilles tolérées)
  ○ OECI feuille « Indicateurs chirurgie » (non consommée) — présente
  ○ OECI feuille « Origine géo » (non consommée) — présente
  ○ OECI feuille « Effectifs recherche » (non consommée) — présente
  

## 4. Contrôle 2 — Contrat de `donnees.csv`

`pandas` seul : colonnes exactes, domaines des dimensions, vocabulaire des `variable`,
clé d'unicité, valeurs numériques, bornes (survie ∈ [0,100], délais ≥ 0, comptes ≥ 0),
années ∈ [2022, 2025].

In [5]:
def controler_donnees_csv(chemin=DATA_DIR / 'donnees.csv'):
    res = []
    df = pd.read_csv(chemin, dtype={'stade': 'string'})
    attendu = {'annee', 'source', 'niveau', 'entite', 'appareil', 'organe',
               'age', 'stade', 'population', 'variable', 'valeur'}
    cols = set(df.columns)
    res.append((cols == attendu, 'colonnes exactes'
                + ('' if cols == attendu else f" — manquantes {attendu-cols} / en trop {cols-attendu}")))

    def domaine(col, dom):
        obs = set(df[col].dropna().unique())
        res.append((obs <= dom, f"{col} ⊆ {sorted(dom)}"
                    + ('' if obs <= dom else f" — HORS {sorted(obs - dom)}")))

    domaine('source', {'BN', 'DIM APHP', 'EDS APHP'})
    domaine('niveau', {'aphp', 'ghu', 'hopital', 'type_etab'})
    domaine('age', {'tous', 'pédiatrie', 'adultes'})
    domaine('population', {'tous', 'nouveaux'})
    domaine('stade', {'I-III', 'IV'})
    voc = {'nb_patients', 'nb_sejours_chirurgie', 'nb_sejours_chimiotherapie',
           'nb_sejours_radiotherapie', 'nb_sejours_palliatifs', 'delai_global_median',
           'delai_chirurgie_median', 'delai_traitement_medical_median', 'delai_radio_median',
           'nb_patients_stade', 'survie_1an', 'survie_5ans'}
    domaine('variable', voc)

    dimk = ['annee', 'source', 'niveau', 'entite', 'appareil', 'organe',
            'age', 'stade', 'population', 'variable']
    ndup = int(df.duplicated(subset=dimk).sum())
    res.append((ndup == 0, 'clé unique (0 doublon)' + ('' if ndup == 0 else f" — {ndup} doublons")))

    v = pd.to_numeric(df['valeur'], errors='coerce')
    nnum = int((v.isna() & df['valeur'].notna()).sum())
    res.append((nnum == 0, 'valeur numérique (hors NaN légitimes)'
                + ('' if nnum == 0 else f" — {nnum} non numériques")))

    sv = v[df.variable.isin(['survie_1an', 'survie_5ans'])]
    res.append((bool(sv.between(0, 100).all()), f"survie dans [0,100]  (min={sv.min():.0f} max={sv.max():.0f})"))
    dl = v[df.variable.str.startswith('delai_')]
    res.append((bool((dl >= 0).all()), f"délais >= 0  (min={dl.min():.1f})"))
    ct = v[df.variable.str.startswith('nb_')]
    res.append((bool((ct >= 0).all()), f"comptes >= 0  (min={ct.min():.0f})"))

    an = pd.to_numeric(df['annee'], errors='coerce')
    hors = sorted(set(an[~an.between(2022, 2025)].dropna().astype(int)))
    res.append((not hors, 'annee dans [2022, 2025]' + ('' if not hors else f" — hors {hors}")))
    return res

_csv = controler_donnees_csv()
print(f"Contrôle 2 — {len(_csv)} vérification(s) :")
for ok, lib in _csv:
    print(f"  {_tag(ok)} {lib}")
brancher(_csv)

# Concordance hôpitaux survie <-> délais (informatif : noms diffèrent parfois en réel)
h_surv = set(long[(long.source == 'EDS APHP') & (long.niveau == 'hopital')].entite)
h_del = set(long[(long.source == 'DIM APHP') & (long.niveau == 'hopital')
                 & (long.variable.str.startswith('delai_'))].entite)
recouv = len(h_surv & h_del) / max(1, len(h_surv | h_del))
check(f"concordance hôpitaux survie <-> délais ({recouv:.0%} ; {len(h_surv)} vs {len(h_del)})",
      recouv >= 0.8, critique=False)

Contrôle 2 — 13 vérification(s) :
  ✓ colonnes exactes
  ✓ source ⊆ ['BN', 'DIM APHP', 'EDS APHP']
  ✓ niveau ⊆ ['aphp', 'ghu', 'hopital', 'type_etab']
  ✓ age ⊆ ['adultes', 'pédiatrie', 'tous']
  ✓ population ⊆ ['nouveaux', 'tous']
  ✓ stade ⊆ ['I-III', 'IV']
  ✓ variable ⊆ ['delai_chirurgie_median', 'delai_global_median', 'delai_radio_median', 'delai_traitement_medical_median', 'nb_patients', 'nb_patients_stade', 'nb_sejours_chimiotherapie', 'nb_sejours_chirurgie', 'nb_sejours_palliatifs', 'nb_sejours_radiotherapie', 'survie_1an', 'survie_5ans']
  ✓ clé unique (0 doublon)
  ✓ valeur numérique (hors NaN légitimes)
  ✓ survie dans [0,100]  (min=8 max=99)
  ✓ délais >= 0  (min=15.2)
  ✓ comptes >= 0  (min=0)
  ✓ annee dans [2022, 2025]


## 5. Synthèse

Tests **critiques** rouges → `AssertionError` (bloque le build). Tests **informatifs**
(⚠) et **non vérifiés** (○, fichier absent) n'interrompent pas.

In [6]:
for nom, ok, detail, crit in tests:
    print(f"  {_tag(ok, crit)} {nom}" + (f"  ({detail})" if detail else ''))

ko = [t for t in tests if t[1] is False and t[3]]
warn = [t for t in tests if t[1] is False and not t[3]]
skip = [t for t in tests if t[1] is None]
verts = sum(1 for t in tests if t[1] is True)
print(f"\n— {len(tests)} test(s) : {verts} OK, {len(ko)} critiques KO, "
      f"{len(warn)} avertissements, {len(skip)} non vérifiés")
msg = 'TOUS LES TESTS CRITIQUES SONT VERTS' if not ko else f'{len(ko)} TEST(S) CRITIQUE(S) EN ECHEC'
print(msg)
assert not ko, 'Contrôle pré-prod : test(s) critique(s) en échec -> build BLOQUE'

  ✓ OECI : fichier réel = indicateurs_oeci_2025_M12.xlsx
  ✓ OECI « Indicateurs patient » : libellés de dimensions
  ✓ OECI « Indicateurs patient » : modalités Niveau reconnues
  ✓ OECI « Indicateurs patient » : forme GHU = long
  ✓ OECI « Indicateurs séjour » : libellés de dimensions
  ✓ OECI « Indicateurs séjour » : modalités Niveau reconnues
  ✓ OECI « Indicateurs séjour » : forme GHU = long
  ✓ OECI « Survie globale » : libellés de dimensions
  ✓ OECI « Survie globale » : modalités Niveau reconnues
  ✓ OECI « Survie globale » : forme GHU = court
  ✓ OECI « Délais PEC » : libellés de dimensions
  ✓ OECI « Délais PEC » : modalités Niveau reconnues
  ✓ OECI « Délais PEC » : forme GHU = long
  ✓ OECI « Délais PEC » : 4 blocs de délais (coquilles tolérées)
  ○ OECI feuille « Indicateurs chirurgie » (non consommée) — présente
  ○ OECI feuille « Origine géo » (non consommée) — présente
  ○ OECI feuille « Effectifs recherche » (non consommée) — présente
  ○ Régional (canceroBR_*_Pat_*.xlsx

## 6. Exploration libre

Cellule libre pour inspecter `long` (aucun import de `src/`).

In [7]:
ap = long[(long.source == 'DIM APHP') & (long.entite == 'AP-HP')
          & (long.appareil == 'TOTAL') & (long.organe == 'TOTAL')
          & (long.variable.isin(['nb_patients', 'delai_global_median']))
          & (long.population == 'tous')]
ap.pivot_table(index='annee', columns='variable', values='valeur').reset_index()

variable,annee
